# Abstraction

> Generated from [`05-Abstraction.py`](05-Abstraction.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Hiding implementation complexity behind a simple interface — the difference from encapsulation, and using `abc.ABC`/`@abstractmethod` to actually enforce that subclasses implement required methods.

### Key points
- **Abstraction** hides *how* something works and exposes only *what* it does (e.g. driving a car: you use the steering wheel/pedals, not the engine internals). This is different from **encapsulation**, which hides *data*.
- Without enforcement, a base class method that's just `pass` doesn't force subclasses to override it — a forgotten override silently does nothing until it's called and causes a bug.
- **`ABC` + `@abstractmethod`** (from the `abc` module) fixes this: the abstract class can't be instantiated directly, and any subclass missing a required method also can't be instantiated.
- Abstract classes can still have **concrete (shared) methods** alongside abstract ones — subclasses get the shared logic for free while being forced to fill in the abstract parts.
- **`ABC.register(SomeClass)`**: makes `SomeClass` count as a "virtual subclass" for `isinstance()`/`issubclass()` checks, without any real inheritance — useful for retrofitting an interface onto a class you don't control.
- Stacking `@property` above `@abstractmethod` forces subclasses to expose the member as a *property* (attribute-style access), not a plain method call.
- **`typing.Protocol`** is structural typing — any class with matching methods satisfies the protocol automatically, no inheritance or registration needed (contrast with ABC's nominal typing, which requires an explicit `is-a` declaration).

### Gotchas (the part worth revising twice)
- **This is deliberate fail-fast design**: without ABC, a missing method surfaces only when it's called (possibly deep in production, much later). With ABC, the `TypeError` happens the moment you try to instantiate the incomplete class — far cheaper to debug.
- `Flyer.register(Airplane)` makes `isinstance(plane, Flyer)` return `True` and `issubclass(Airplane, Flyer)` return `True`, but `Flyer` is NOT in `Airplane.__mro__` — there's no real inheritance link, no shared code, no MRO change. It's purely a registered relationship for type-checking purposes.
- ABC = nominal typing (must explicitly inherit or register to "count"). Protocol = structural typing (matching shape is enough). Don't confuse the two — `isinstance()` against a plain `Protocol` only works if it's marked `@runtime_checkable`.
- An abstract class can still define `__init__` and other regular methods — "abstract" only blocks instantiation of the class itself and requires the specific `@abstractmethod`-marked methods to be overridden, not the whole class to be rewritten.

### When it matters
Use ABC when you're designing a shared interface that multiple implementations must honor (payment methods, shape types, notification channels) and you want a hard guarantee — not just a comment — that every implementation provides the required methods.

### One-line recap example
```python
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...

# Shape() -> TypeError: can't instantiate abstract class
```

In [ ]:
from abc import ABC, abstractmethod

## 01. Abstraction - Introduction

- Abstraction hides implementation details and exposes only what's
  necessary - a simplified interface for using an object.
- Distinct from encapsulation (hiding data) - abstraction is about
  hiding complexity of HOW something works, exposing only WHAT it does.

## 02. Real-World Analogy

## 03. Abstraction via Regular Classes (informal)

In [ ]:
class Car:
    def start(self):
        self._ignite_engine()  # internal detail, hidden behind start()
        print("Car started")

    def _ignite_engine(self):
        print("(internal) igniting engine, checking fuel injection...")

print("\n# 03. Abstraction via Regular Classes")
car = Car()
car.start()  # simple interface - complexity hidden inside

## 04. The Problem: Nothing Forces Subclasses to Implement Anything

In [ ]:
class PaymentMethod:
    def pay(self, amount):
        pass  # subclasses SHOULD override this, but nothing enforces it

class BrokenPayment(PaymentMethod):
    pass  # forgot to implement pay() - no error until you call it and fail silently

print("\n# 04. The Problem Without Enforcement")
broken = BrokenPayment()
print("pay() returns:", broken.pay(100))  # silently does nothing - a real bug risk

## 05. Abstract Base Classes (ABC) - Enforcing an Interface

- ABC + @abstractmethod makes a class impossible to instantiate directly,
  and forces subclasses to implement the marked methods.

In [ ]:
class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass

print("\n# 05. Abstract Base Classes (ABC)")
try:
    Shape()  # cannot instantiate an abstract class directly
except TypeError as e:
    print("Caught:", e)

## 06. Implementing an Abstract Class

In [ ]:
class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

    def perimeter(self):
        return 2 * 3.14159 * self.radius

print("\n# 06. Implementing an Abstract Class")
shapes = [Rectangle(4, 5), Circle(3)]
for shape in shapes:
    print(f"{shape.__class__.__name__}: area={shape.area():.2f}, perimeter={shape.perimeter():.2f}")

## 07. Forgetting to Implement an Abstract Method - Caught Immediately

In [ ]:
class IncompleteShape(Shape):
    def area(self):
        return 0
    # perimeter() not implemented

print("\n# 07. Forgetting to Implement an Abstract Method")
try:
    IncompleteShape()
except TypeError as e:
    print("Caught:", e)  # error at instantiation, not silently ignored

## 08. Abstract Classes Can Still Have Concrete (shared) Methods

In [ ]:
class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        pass

    def describe(self):  # concrete method - shared by all subclasses
        return f"{self.name} earns {self.calculate_pay()}"

class SalariedEmployee(Employee):
    def calculate_pay(self):
        return 5000

print("\n# 08. Abstract Classes with Concrete Methods")
emp = SalariedEmployee("Debanjan")
print(emp.describe())

## 09. Abstraction vs Encapsulation (recap)

## 10. Virtual Subclass Registration (ABC.register())

- `SomeABC.register(SomeClass)` makes SomeClass "count as" a subclass of
  SomeABC for isinstance()/issubclass() checks, WITHOUT actually inheriting
  from it - no shared code, no MRO change, just a registered relationship.
- Useful for retrofitting an interface onto a class you don't control (e.g.
  a third-party class) or don't want to touch.

In [ ]:
class Flyer(ABC):
    @abstractmethod
    def fly(self):
        pass

class Airplane: # does NOT inherit from Flyer at all
    def fly(self):
        return "Airplane flying"

Flyer.register(Airplane) # registers Airplane as a "virtual" subclass of Flyer

print("\n# 10. Virtual Subclass Registration (ABC.register())")
plane = Airplane()
print("isinstance(plane, Flyer):", isinstance(plane, Flyer)) # True, despite no inheritance
print("issubclass(Airplane, Flyer):", issubclass(Airplane, Flyer))
print("Flyer in Airplane.__mro__:", Flyer in Airplane.__mro__) # False - no real inheritance link

## 11. @abstractmethod Combined with @property (Abstract Property)

- Stacking @property above @abstractmethod forces subclasses to provide a
  PROPERTY (not a regular method) with that name - enforces both "must
  implement this" and "must expose it as an attribute, not a method call".

In [ ]:
class Vehicle(ABC):
    @property
    @abstractmethod
    def max_speed(self):
        pass

class SportsCar(Vehicle):
    @property
    def max_speed(self): # must also be a property, not a plain method
        return 300

print("\n# 11. Abstract Property")
car = SportsCar()
print("car.max_speed (accessed like an attribute, no parentheses):", car.max_speed)
try:
    Vehicle() # still can't instantiate the abstract class itself
except TypeError as e:
    print("Caught:", e)

## 12. Protocol vs ABC - Structural vs Nominal Typing

- ABC (nominal typing): a class must explicitly inherit from the ABC (or be
  registered) to "count" - the relationship is declared by name.
- typing.Protocol (structural typing): ANY class with matching methods
  automatically satisfies the protocol, no inheritance or registration
  needed - "if it has the right shape, it counts" (duck typing, checkable
  by type checkers). Full depth on typing lives in the standard library file.

In [ ]:
from typing import Protocol

class Flappable(Protocol):
    def flap(self) -> str: ...

class Bird: # never inherits from Flappable, never registers - just has the method
    def flap(self):
        return "Bird flapping wings"

print("\n# 12. Protocol vs ABC (structural vs nominal typing)")
print("# - ABC: needs explicit inheritance/registration (nominal).")
print("# - Protocol: matching methods are enough, no inheritance needed (structural).")
bird = Bird()
print("isinstance(bird, Flappable) works only if Protocol is @runtime_checkable - see typing docs.")
print("bird.flap():", bird.flap())

## 13. Why You Can't Instantiate a Class with Missing Abstract Methods

- This is deliberate fail-fast design: the error happens the moment you try
  to CREATE the incomplete object, not later when the missing method
  happens to get called (which might be much later, in production).
- Catching the mistake at instantiation is far cheaper than debugging a
  silent no-op or an AttributeError deep inside unrelated code.

In [ ]:
class Task(ABC):
    @abstractmethod
    def run(self):
        pass

class BrokenTask(Task):
    pass # never implements run() - and never will, as written

print("\n# 13. Fail-Fast: Can't Instantiate with Unimplemented Abstract Methods")
try:
    BrokenTask() # fails HERE, even though run() is never called
except TypeError as e:
    print("Caught immediately at instantiation, before run() would ever be called:", e)